In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split,regexp_replace,col,lit,expr
from delta.tables import DeltaTable
spark =  SparkSession.builder\
    .getOrCreate()

########################################
############_Transformation#############
#########################################

def transform(df):

    df=df.withColumn("name_parts",split(col("customer_name"), ","))\
        .withColumn("first_name", expr("get(name_parts, 1)")) \
        .withColumn("last_name", expr("get(name_parts, 0)"))\
        .drop("name_parts")\
        .withColumn("postcode",regexp_replace(col("postcode"),"\\.0$","") )\
        .withColumn("country",lit("USA"))\
        .drop("customer_name","file_path")


    df =df.dropDuplicates(["customer_id"])

    df = df.select(
    'customer_id','first_name',
    'last_name',
    'tax_id',
    'tax_code',
    'state',
    'city',
    'postcode',
    'street',
    'number',
    'unit',
    'region',
    'district',
    'country',
    'lon',
    'lat',
    'ship_to_address',
    'valid_from',
    'valid_to',
    'units_purchased',
    'loyalty_segment',
    'last_updates_ts'
    )
    return df

def scd_merge_table(spark,source_table, target_table, business_key):

    source_df = spark.table(source_table)

    if not spark.catalog.tableExists(target_table):
        print("first load: creating silvr table")
        source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    else:
        print("ïncremental load: performing SCD type 1 merge")

        delta_table=DeltaTable.forName(spark,target_table)
        merge_condition = " AND ".join(
            [f"target.{col}=source.{col}" for col in business_key]
        )
        
        delta_table.alias("target").merge(source_df.alias("source"), merge_condition
                                        ).whenMatchedUpdateAll()\
                                            .whenNotMatchedInsertAll()\
                                                .execute()
        print("merge successfully complted")
########################################
############_Main Logic#################
#########################################
source_table="ecommerce_analytics.bronze.customers"
target_table = "ecommerce_analytics.silver.customers"
business_key = ["customer_id"]
df=spark.read.table(source_table)

transformed_df = transform(df)
transformed_df.createOrReplaceTempView("temp_source")

scd_merge_table(spark,"temp_source",target_table,business_key)
